In [40]:
import pandas as pd
import numpy as np
import pickle
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier, StackingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from imblearn.over_sampling import SMOTE

In [41]:
df = pd.read_csv('data/final_data_wine.csv', index_col=0)

In [42]:
X = df.drop(columns=['type'])
y = df['type']

In [43]:
X.to_csv("data/test.csv")

In [44]:
y.to_csv("data/relust_test.csv")

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)

In [46]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [47]:
models = {}

In [48]:
ml1 = LogisticRegression(max_iter=1000, random_state=42)
ml1.fit(X_train_scaled, y_train)
models['ML1_LogReg'] = ml1

In [49]:
ml2 = GradientBoostingClassifier(n_estimators=100, random_state=42)
ml2.fit(X_train_scaled, y_train)
models['ML2_GradientBoosting'] = ml2

In [50]:
ml3 = CatBoostClassifier(iterations=100, verbose=0, random_state=42)
ml3.fit(X_train_scaled, y_train)
models['ML3_CatBoost'] = ml3

In [51]:
ml4 = BaggingClassifier(estimator=LogisticRegression(max_iter=1000), 
                        n_estimators=50, random_state=42)
ml4.fit(X_train_scaled, y_train)
models['ML4_Bagging'] = ml4

In [52]:
base_estimators = [
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('svc', SVC(kernel='linear', probability=True, random_state=42))
]
ml5 = StackingClassifier(estimators=base_estimators, 
                         final_estimator=LogisticRegression(),
                         cv=3)
ml5.fit(X_train_scaled, y_train)
models['ML5_Stacking'] = ml5

In [53]:
ml6 = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
ml6.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
ml6.fit(X_train_scaled, y_train, epochs=30, validation_split=0.2, verbose=0)
models['ML6_NeuralNet'] = ml6

c:\Users\danii\Desktop\git\master\mlbd\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [54]:
for name in ['ML1_LogReg', 'ML2_GradientBoosting', 'ML4_Bagging', 'ML5_Stacking']:
    with open(f"models/{name}.pkl", "wb") as f:
        pickle.dump(models[name], f)

models['ML3_CatBoost'].save_model("models/ML3_CatBoost.cbm")
models['ML6_NeuralNet'].save("models/ML6_NeuralNet.keras")

In [55]:
for name, model in models.items():
    if name == 'ML6_NeuralNet':
        probs = model.predict(X_test_scaled, verbose=0)
        y_pred = (probs > 0.5).astype(int).ravel()
    else:

        y_pred = model.predict(X_test_scaled).ravel()
        
    f1 = f1_score(y_test, y_pred)
    print(f"{name}: f1 = {f1:.4f}")

ML1_LogReg: f1 = 0.9930
ML2_GradientBoosting: f1 = 0.9894
ML3_CatBoost: f1 = 0.9894
ML4_Bagging: f1 = 0.9912
ML5_Stacking: f1 = 0.9930
ML6_NeuralNet: f1 = 0.9947


In [56]:
import keras

MODEL_PATHS = {
    "ML1: Классическая (kNN)": "models/ML1_LogReg.pkl",
    "ML2: Бустинг": "models/ML2_GradientBoosting.pkl",
    "ML3: CatBoost": "models/ML3_CatBoost.cbm",
    "ML4: Бэггинг": "models/ML4_Bagging.pkl",
    "ML5: Стэкинг": "models/ML5_Stacking.pkl",
    "ML6: Нейросеть": "models/ML6_NeuralNet.keras"
}

def load_models():
    loaded_models = {}
    
    for name, path in MODEL_PATHS.items():
        if path.endswith('.keras'):
            loaded_models[name] = keras.models.load_model(path)
        elif path.endswith('.cbm'):
            import catboost
            loaded_models[name] = catboost.CatBoostClassifier()
            loaded_models[name].load_model(path)
        else:
            with open(path, 'rb') as f:
                loaded_models[name] = pickle.load(f)
    
    return loaded_models

models = load_models()